<a href="https://colab.research.google.com/github/zm-practice/InstructABSA/blob/main/Notebooks/ATE_Training_%26_Inference_rest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries

In [15]:
import torch
import transformers
import datasets
import platform

print("=== 环境信息 ===")
print(f"Python版本: {platform.python_version()}")
print(f"PyTorch版本: {torch.__version__}")
print(f"Transformers版本: {transformers.__version__}")
print(f"Datasets版本: {datasets.__version__}")
print(f"CUDA版本: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

=== 环境信息 ===
Python版本: 3.12.13
PyTorch版本: 2.10.0+cu128
Transformers版本: 5.0.0
Datasets版本: 4.0.0
CUDA版本: 12.8
GPU: Tesla T4
显存: 14.6 GB


In [1]:
# 修复 utils.py 里的 tokenizer 参数
utils_path = '/content/drive/MyDrive/InstructABSA-main/InstructABSA/utils.py'

with open(utils_path, 'r') as f:
    content = f.read()

# 把 tokenizer= 改成 processing_class=
content = content.replace('tokenizer=self.tokenizer', 'processing_class=self.tokenizer')

with open(utils_path, 'w') as f:
    f.write(content)

print("修复完成")

修复完成


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 列出你Drive里的文件夹
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['校赛A.docx', 'Google AI Studio', 'Resume.gdoc', 'SemEval14', 'InstructABSA-main']


In [3]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    IN_COLAB = True
except:
    IN_COLAB = False

Mounted at /content/drive


In [4]:
if IN_COLAB:
  !pip install transformers
  !pip install datasets
  !pip install evaluate
  !pip install sentencepiece

In [5]:
import os
import torch

if IN_COLAB:
    root_path = '/content/drive/MyDrive/InstructABSA-main'
else:
    root_path = 'Enter local path'

use_mps = True if torch.has_mps else False
os.chdir(root_path)

/tmp/ipykernel_16988/1406701113.py:9: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  use_mps = True if torch.has_mps else False


In [6]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from instructions import InstructionsHandler

## Training

In [7]:
task_name = 'ate'
#experiment_name = 'lapt2014_iabsa1'
experiment_name = 'lapt_rest_iabsa2_flan'
#model_checkpoint = 'allenai/tk-instruct-base-def-pos'
model_checkpoint = 'google/flan-t5-base'
print('Experiment Name: ', experiment_name)
model_out_path = './Models'
model_out_path = os.path.join(model_out_path, task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Model output path: ', model_out_path)

Experiment Name:  lapt_rest_iabsa2_flan
Model output path:  ./Models/ate/googleflan-t5-base-lapt_rest_iabsa2_flan


# 尝试联合数据，lapt+rest

In [8]:
# ── 数据加载（正确方式）──────────────────────────
id_tr_df  = pd.read_csv('./Dataset/SemEval14/Train/Laptops_Train.csv')
id_te_df  = pd.read_csv('./Dataset/SemEval14/Test/Laptops_Test.csv')
ood_tr_df = pd.read_csv('./Dataset/SemEval14/Train/Restaurants_Train.csv')
ood_te_df = pd.read_csv('./Dataset/SemEval14/Test/Restaurants_Test.csv')

# ── 指令配置 ────────────────────────────────────
instruct_handler = InstructionsHandler()
instruct_handler.load_instruction_set2()  # ← 改这里切换指令1/2

# ── 数据格式化 ──────────────────────────────────
loader = DatasetLoader(
    train_df_id=id_tr_df,
    test_df_id=id_te_df,
    train_df_ood=ood_tr_df,
    test_df_ood=ood_te_df
)

# Laptop → bos_instruct1，Restaurant → bos_instruct2
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(
        loader.train_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(
        loader.test_df_id, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct1'],
        instruct_handler.ate['eos_instruct'])
if loader.train_df_ood is not None:
    loader.train_df_ood = loader.create_data_in_ate_format(
        loader.train_df_ood, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct2'],
        instruct_handler.ate['eos_instruct'])
if loader.test_df_ood is not None:
    loader.test_df_ood = loader.create_data_in_ate_format(
        loader.test_df_ood, 'term', 'raw_text', 'aspectTerms',
        instruct_handler.ate['bos_instruct2'],
        instruct_handler.ate['eos_instruct'])

In [9]:
# Create T5 utils object
t5_exp = T5Generator(model_checkpoint)

# Tokenize Dataset
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Training arguments
training_args = {
    'output_dir':model_out_path,
    'eval_strategy':'no',#"epoch"没有验证集,#把 evaluation_strategy 改成 eval_strategy
    'learning_rate':5e-5,
    'lr_scheduler_type':'cosine',
    'per_device_train_batch_size':8,
    'per_device_eval_batch_size':16,
    'num_train_epochs':4,
    'weight_decay':0.01,
    'warmup_ratio':0.1,
    'save_strategy':'no',
    'load_best_model_at_end':False,
    'push_to_hub':False,
    'eval_accumulation_steps':1,
    'predict_with_generate':True,
    #'use_mps_device':use_mps
}

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3041 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [10]:
# Train model，不对训练只传入了 id_tokenized_ds，Restaurant 数据完全没参与训练。
#model_trainer = t5_exp.train(id_tokenized_ds, **training_args)
from datasets import concatenate_datasets, DatasetDict

# 合并 Laptop + Restaurant 的训练集
combined_train = concatenate_datasets([
    id_tokenized_ds['train'],
    ood_tokenized_ds['train']
])

# 测试集只用 Laptop
combined_ds = DatasetDict({
    'train': combined_train,
    'test': id_tokenized_ds['test']
})

# 用合并后的数据训练
model_trainer = t5_exp.train(combined_ds, **training_args)
print('Model saved at: ', model_out_path)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer device: cuda:0

Model training started ....


Step,Training Loss
500,7.134974
1000,4.349987
1500,3.390435
2000,2.884983
2500,2.706888
3000,2.600029


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at:  ./Models/ate/googleflan-t5-base-lapt_rest_iabsa2_flan


## Inference

In [11]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set2()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct1'], instruct_handler.ate['eos_instruct'])

In [12]:
# Model inference - Loading from Checkpoint
t5_exp = T5Generator(model_out_path)

# Tokenize Datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenzed_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get prediction labels - Training set
id_tr_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'train',  batch_size = 16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]
#不支持trained_model_path = model_out_path,参数
# Get prediction labels - Testing set
id_te_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'test',  batch_size = 16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Model loaded to:  cuda


100%|██████████| 191/191 [01:59<00:00,  1.59it/s]


Model loaded to:  cuda


100%|██████████| 50/50 [00:29<00:00,  1.70it/s]


In [13]:
p, r, f1 ,_= t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)#少了一个参数，加入_
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1,_ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

Train Precision:  0.7712634186622626
Train Recall:  0.7157088122605364
Train F1:  0.7424483306836248
Test Precision:  0.7045454545454546
Test Recall:  0.6308139534883721
Test F1:  0.6656441717791411


In [14]:
# 看前5条预测输出
for i, (gt, pred) in enumerate(zip(id_te_labels[:5], id_te_pred_labels[:5])):
    print(f"GT:   {gt}")
    print(f"PRED: {pred}")
    print()

GT:   Boot time
PRED: Boot time

GT:   tech support
PRED: tech service

GT:   noaspectterm
PRED: noaspectterm

GT:   Set up
PRED: aspectterm

GT:   Windows 8, touchscreen functions
PRED: Windows, screen functions

